In [ ]:
! echo $CUDA_VISIBLE_DEVICES

In [ ]:
# !which gcc
# ! echo $SLURM_JOB_ID


!gpustat

In [1]:
# languages='english'
# rewards_in_the_name = 'FR'

# from datasets import Dataset, DatasetDict, load_from_disk
from datasets import load_dataset
import pandas as pd
import evaluate
import json
import re
import numpy as np
from PIL import Image
# import pytesseract
from ddgs import DDGS
from qwen_vl_utils import process_vision_info
from tqdm import tqdm

dataset1 = load_dataset('HF_DATASET/HF_PATH',split='train')

dataset1 = dataset1.add_column("language_encoded", dataset1["Language"])
dataset1 = dataset1.class_encode_column("language_encoded")
dataset1 = dataset1.filter(lambda row: row['Language']=='tamil', num_proc=4)

split_dataset1 = dataset1.train_test_split(test_size=0.3, seed=42, stratify_by_column="language_encoded")


# train_dataset = split_dataset["train"]
# test_dataset = split_dataset["test"]

# train_dataset1 = split_dataset1["train"]
dataset1 = split_dataset1["test"]

# dataset_split = dataset.train_test_split(test_size=0.3, seed=42, stratify_by_column="language_encoded")
# split_dataset = dataset.train_test_split(test_size=0.3, seed=42)
# split_dataset1 = dataset1.train_test_split(test_size=0.3, seed=42, stratify_by_column="language_encoded")


# train_dataset = split_dataset["train"]
# test_dataset = split_dataset["test"]

# train_dataset1 = split_dataset1["train"]
# test_dataset1 = split_dataset1["test"]

# format_rewards = []
# accuracy_rewards=[]
# inconsis_penaltys=[]
# tool_debate_rewards=[]



In [2]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a MCQ question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the correct option. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> correct option here </answer>"
)

In [3]:
import random
from copy import deepcopy

# ---------------------------------------------------------
# Language-specific option label mapping
# ---------------------------------------------------------
OPTION_LABEL_MAP = {
    "english": {
        "a": "a",
        "b": "b",
        "c": "c",
        "d": "d"
    },
    "hindi": {
        "a": "क",
        "b": "ख",
        "c": "ग",
        "d": "घ"
    },
    "bengali": {
        "a": "ক",
        "b": "খ",
        "c": "গ",
        "d": "ঘ"
    },
    "marathi": {
        "a": "क",
        "b": "ख",
        "c": "ग",
        "d": "घ"
    },
    "gujarati": {
        "a": "ક",
        "b": "ખ",
        "c": "ગ",
        "d": "ઘ"
    },
    "tamil": {
        "a": "௧",
        "b": "௨",
        "c": "௩",
        "d": "௪"
    }
}


# ---------------------------------------------------------
# Fisher-Yates shuffle
# ---------------------------------------------------------
def fisher_yates_shuffle(items, rng):
    """
    Unbiased Fisher-Yates shuffle.

    Args:
        items: list of items
        rng: random.Random object

    Returns:
        shuffled list
    """
    shuffled = items.copy()

    for i in range(len(shuffled) - 1, 0, -1):
        j = rng.randint(0, i)
        shuffled[i], shuffled[j] = shuffled[j], shuffled[i]

    return shuffled


# ---------------------------------------------------------
# Normalize final answer
# ---------------------------------------------------------
def resolve_answer_key(example):
    """
    Convert native answer label into standard key: a/b/c/d.

    Example:
        Hindi क -> a
        Bengali খ -> b
        Tamil ௩ -> c
    """

    language = example.get("Language", "").strip().lower()
    native_answer = example.get("Final Answer")

    if native_answer is None:
        return None, None, None

    native_answer = str(native_answer).strip()

    lang_map = OPTION_LABEL_MAP.get(language, {})
    reverse_map = {v: k for k, v in lang_map.items()}

    # Case 1: answer is already a/b/c/d
    if native_answer.lower() in ["a", "b", "c", "d"]:
        option_key = native_answer.lower()

    # Case 2: answer is native label, e.g., क/খ/௩
    else:
        option_key = reverse_map.get(native_answer)

    if option_key is None:
        return native_answer, None, None

    answer_text = example.get(f"Option {option_key}")

    return native_answer, option_key, answer_text


# ---------------------------------------------------------
# Build user input from options
# ---------------------------------------------------------
def build_user_input(example):
    """
    Build the question + options text using language-specific labels.
    """

    language = example.get("Language", "").strip().lower()
    lang_map = OPTION_LABEL_MAP.get(language, OPTION_LABEL_MAP["english"])

    user_input = (
        f"Question: {example['Question']}\n"
        f"Options: "
        f"{lang_map.get('a', 'a')}. {example['Option a']}, "
        f"{lang_map.get('b', 'b')}. {example['Option b']}, "
        f"{lang_map.get('c', 'c')}. {example['Option c']}, "
        f"{lang_map.get('d', 'd')}. {example['Option d']}."
    )

    return user_input


# ---------------------------------------------------------
# Create one formatted row
# ---------------------------------------------------------
def format_one_row(example, shuffle_id=0):
    """
    Format one example into the final prompt/solution structure.
    """

    templist = []

    language = example.get("Language", "").strip().lower()
    lang_map = OPTION_LABEL_MAP.get(language, OPTION_LABEL_MAP["english"])

    native_answer, option_key_letter, answer_text = resolve_answer_key(example)

    user_input = build_user_input(example)

    if example.get("Final Answer") is None:
        sol = f"<think>{example.get('Reasoning', '')}</think><answer>None</answer>"
    else:
        sol = (
            f"<think>{example.get('Reasoning', '')}</think>"
            f"<answer>{native_answer}. {answer_text}</answer>"
        )

    returned_dict = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": SYSTEM_PROMPT
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": user_input,
                }
            ],
        },
    ]

    return {
        "images": templist,
        "item_id": example["item_id"],
        "shuffle_id": shuffle_id,
        "is_original": shuffle_id == 0,
        "prompt": returned_dict,
        "question": user_input,
        "solution": sol,
        "native_answer_label": native_answer,
        "option_key": option_key_letter,
        "answer_text": answer_text,
        "Option a": example.get("Option a"),
        "Option b": example.get("Option b"),
        "Option c": example.get("Option c"),
        "Option d": example.get("Option d"),
        "Final Answer": example.get("Final Answer"),
        "Language": example.get("Language"),
    }


# ---------------------------------------------------------
# Shuffle one MCQ example
# ---------------------------------------------------------
def create_shuffled_example(example, shuffle_id, seed=42):
    """
    Create one shuffled version of an MCQ.

    The same item_id is kept.
    Options are shuffled.
    Final Answer is updated to the new native label.
    """

    rng = random.Random(seed + int(example["item_id"]) * 100 + shuffle_id)

    new_example = deepcopy(example)

    language = example.get("Language", "").strip().lower()
    lang_map = OPTION_LABEL_MAP.get(language, OPTION_LABEL_MAP["english"])

    native_answer, old_option_key, old_answer_text = resolve_answer_key(example)

    if old_option_key is None:
        return new_example

    original_options = [
        ("a", example.get("Option a")),
        ("b", example.get("Option b")),
        ("c", example.get("Option c")),
        ("d", example.get("Option d")),
    ]

    # Fisher-Yates shuffle
    shuffled_options = fisher_yates_shuffle(original_options, rng)

    new_option_keys = ["a", "b", "c", "d"]

    # Assign shuffled option texts to Option a/b/c/d
    for new_key, (_, option_text) in zip(new_option_keys, shuffled_options):
        new_example[f"Option {new_key}"] = option_text

    # Find where the original correct answer text moved
    new_correct_key = None

    for new_key in new_option_keys:
        if new_example[f"Option {new_key}"] == old_answer_text:
            new_correct_key = new_key
            break

    if new_correct_key is None:
        raise ValueError(
            f"Could not find correct answer text after shuffling for item_id={example.get('item_id')}"
        )

    # Convert new correct key to native label
    new_native_answer = lang_map[new_correct_key]
    new_example["Final Answer"] = new_native_answer

    return new_example


# ---------------------------------------------------------
# Main function: produce 1 original + 3 shuffled rows
# ---------------------------------------------------------
def format_data1(example, num_shuffles=3, seed=42):
    """
    For each original MCQ, create 4 rows:
        1 original row
        3 shuffled rows

    Returns:
        list of dictionaries
    """

    all_rows = []

    # Original row
    original_row = format_one_row(
        example=example,
        shuffle_id=0
    )
    all_rows.append(original_row)

    # Shuffled rows
    for shuffle_id in range(1, num_shuffles + 1):
        shuffled_example = create_shuffled_example(
            example=example,
            shuffle_id=shuffle_id,
            seed=seed
        )

        shuffled_row = format_one_row(
            example=shuffled_example,
            shuffle_id=shuffle_id
        )

        all_rows.append(shuffled_row)

    return all_rows

In [6]:
dataset12=[format_data1(example) for example in dataset1]

In [7]:
# print(dataset[0][2])

In [8]:
dataset = []
for i in dataset12:
    for j in i:
        dataset.append(j)



In [ ]:
dataset[5]

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration,AutoProcessor,AutoTokenizer 
# checkpoint_path = './R4_text-Qwen2.5-VL-7B-mcq_ben/checkpoint-660'
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
# processor = AutoProcessor.from_pretrained(mcheckpoint_id,use_fast=True, padding_side="left")
processor = AutoProcessor.from_pretrained(model_id,use_fast=True, padding_side="left")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    # checkpoint_path,
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    # use_cache= False
)

In [11]:
import torch

@torch.inference_mode()
def batch_infer_from_messages(records, model, processor, batch_size=16, max_new_tokens=1024):
    model.eval()
    all_outputs = []

    for start in tqdm(range(0, len(records), batch_size),desc="Batch inference",unit="batch"):
        batch = records[start:start + batch_size]
        # print('1')
        texts = []
        image_inputs = []
        for ex in batch:
            msgs = ex["prompt"]
            texts.append(processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

            # imgs, _ = process_vision_info(msgs)   # returns list of images for that sample
            # image_inputs.append(imgs)

        inputs = processor(
            text=texts,
            # images=image_inputs,
            padding=True,
            return_tensors="pt",
        ).to("cuda")

        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        outputs = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        all_outputs.extend(outputs)

    return all_outputs


In [ ]:
preds = batch_infer_from_messages(dataset, model, processor,16)

In [ ]:
!gpustat

In [16]:
# import json

rowstw = []
for item, pred in zip(dataset, preds):
    rowstw.append({
        "item_id":json.dumps(item["item_id"], ensure_ascii=False),
        "shuffle_id":json.dumps(item["shuffle_id"], ensure_ascii=False),
        "prompt": json.dumps(item["question"], ensure_ascii=False),
        "solution": item["solution"],
        "predictions": pred,
    })


In [17]:
df = pd.DataFrame(rowstw)
df.to_csv("tam_base_qwenVL3.csv", index=False, encoding="utf-8-sig")


In [ ]:
!gpustat